# Assignment 1: Build, Train, and Evaluate Your First Neural Network

**Unit:** Foundations of Deep Learning | **Companion reading:** Lessons 1-3  
**Suggested time:** 2-3 hours

---

## What is this assignment trying to do?

You have likely heard about deep neural networks and image classifiers. This assignment shows **what happens underneath** when a model learns to recognize complex patterns from raw visual pixels.

In six steps you will:

1. **Load and inspect** a real dataset of handwritten digits (MNIST).
2. **Split** data into training, validation, and test sets to avoid data leakage and overfitting.
3. **Build** a Multi-Layer Perceptron (MLP) architecture using PyTorch.
4. **Train** your network using gradient descent and track performance over time.
5. **Run controlled experiments** that change one setting at a time.
6. **Evaluate** your final model and reflect on what the experiments showed.

**Big idea:** Neural networks are collections of layers that transform raw inputs into target predictions by iteratively updating weights to minimize loss.

| Part | What you do | Why |
|------|-------------|-----|
| 1 | Load and visualize MNIST | Inspect your input data before training |
| 2 | Split data & set hyperparameters | Practice proper dataset split strategy |
| 3 | Complete PyTorch MLP architecture | Understand shape transformations and linear layers |
| 4 | Run training loop | Observe loss decrease and accuracy climb |
| 4.5 | Four controlled experiments | Change one variable at a time and measure the effect |
| 5 | Evaluate on test set & reflect | Measure true generalization performance |

### Submission checklist

- [ ] Every TODO is complete (no `None` or `___` left in the notebook)
- [ ] Every cell has been run top to bottom without errors
- [ ] Every `*_answer` and `*_reflection` variable contains your writing
- [ ] The final cell was run and produced **`results.json`**
- [ ] You upload **both** the `.ipynb` file **and** `results.json` to Gradescope


---
## Key Terms (Read this first)

| Term | Plain-English Meaning |
|------|------------------------|
| **MNIST** | Benchmark dataset of 70,000 grayscale images ($28 \times 28$ pixels) of handwritten digits (0-9). |
| **Tensor** | PyTorch's primary data structure (multi-dimensional array) optimized for GPU operations. |
| **Flattening** | Unrolling a 2D image matrix ($28 \times 28$) into a 1D vector (784 numbers). |
| **Batch Size** | Number of image examples processed before updating the model's weights once. |
| **Learning Rate** | Step size multiplier used by gradient descent when updating weights. |
| **Epoch** | One complete pass through every training example in the dataset. |
| **Validation Set** | Data set aside during development to evaluate hyperparameters without cheating on the test set. |
| **Overfitting** | When a model learns training data noise and performs poorly on unseen validation data. |
| **Cross Entropy Loss** | Measure of error for multi-class classification tasks. **Lower is better.** |
| **Ablation** | Deliberately removing one component of a system to measure how much it contributed. |

**Analogy:** Training set = homework practice problems. Validation set = mid-term practice exam used to change study habits. Test set = final exam taken once at the end.


---
## Part 0: Environment Setup

Run the cell below to load PyTorch, Torchvision, Matplotlib, and set up your execution device (CPU vs GPU).

In [ ]:
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


---
## Part 1: Download & Inspect Data

Before building models, machine learning engineers spend time understanding raw data format and image scaling.

In [ ]:
# Data Preprocessing: Convert PIL images to Tensors and normalize pixel intensities
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f'Raw training examples: {len(full_dataset)}')
print(f'Test examples: {len(test_dataset)}')

# Display sample images
fig, axes = plt.subplots(figsize=(10, 4), ncols=5)
for i in range(5):
    image, label = full_dataset[i]
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(f'Label: {label}')
    axes[i].axis('off')
plt.show()


### Part 1 Reflection

**Question:** Why do we normalize pixel intensities (mean=0.1307, std=0.3081) instead of using raw pixel values from 0 to 255?

Write your answer in the next cell (2-4 sentences).

In [ ]:
part1_reflection = ''  # TODO: Write your reflection response here


---
## Part 2: Hyperparameters & Data Splitting

### TODO 1: Choose Starter Hyperparameters

Hyperparameters are settings you choose *before* training starts.

Set reasonable starting values based on the hints in comments below.

In [ ]:
# TODO 1: Replace None with starter hyperparameter values
BATCH_SIZE = None      # Hint: Common choice between 32 and 128 (e.g., 64)
LEARNING_RATE = None   # Hint: Typical initial choice for Adam is 0.001
NUM_EPOCHS = None      # Hint: Set between 3 and 5 for fast training
HIDDEN_DIM = None      # Hint: Number of hidden units (e.g., 128)

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1


In [ ]:
if None in (BATCH_SIZE, LEARNING_RATE, NUM_EPOCHS, HIDDEN_DIM):
    raise ValueError('Please complete TODO 1 before running data loader splitting.')

train_size = int(TRAIN_RATIO * len(full_dataset))
val_size = int(VAL_RATIO * len(full_dataset))
holdout_size = len(full_dataset) - train_size - val_size

# A fixed generator makes the split identical every time the notebook is rerun,
# so your experiment comparisons are not contaminated by a different split.
split_generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset, holdout_dataset = random_split(
    full_dataset, [train_size, val_size, holdout_size], generator=split_generator
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Held out, unused: {len(holdout_dataset)}')
print(f'Test samples: {len(test_dataset)}')


--- 
## Part 3: Build a Neural Network Architecture

### Network Structure
Input ($28 \times 28 = 784$ pixels) $\rightarrow$ Linear Layer $\rightarrow$ ReLU $\rightarrow$ Linear Layer $\rightarrow$ ReLU $\rightarrow$ Output (10 classes)

### TODO 2: Fill in the layer dimensions

Complete the layer dimensions in `__init__` below.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            # TODO 2a: Fill in input dimensions (28*28) and output dimension (hidden_dim)
            nn.Linear(in_features=___, out_features=___),
            nn.ReLU(),
            # Hidden layer
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            # TODO 2b: Fill in input dimension (hidden_dim) and final classification output dimension (10)
            nn.Linear(in_features=___, out_features=___)
        )

    def forward(self, x):
        return self.network(x)

model = MLP(HIDDEN_DIM).to(device)
print(model)


---
## Part 4: Train and Validate Model

Evaluation function provided below calculates total loss and accuracy across a DataLoader.

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total


### TODO 3: Execute Training Loop

Run the training procedure. Loss functions and optimizers handle backpropagation automatically.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_accs = []
val_accs = []

print('Starting Model Training...')
for epoch in range(NUM_EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # 1. Clear previous gradients
        optimizer.zero_grad()

        # 2. Forward pass: compute predictions
        outputs = model(images)

        # 3. Calculate Loss
        loss = criterion(outputs, labels)

        # 4. Backward pass: compute gradients
        loss.backward()

        # 5. Update model weights
        optimizer.step()

    train_loss, train_acc = evaluate(model, train_loader, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')


In [ ]:
# Plotting Learning Curves
plt.figure(figsize=(8, 5))
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Learning Curves')
plt.legend()
plt.grid(True)
plt.show()


---
## Part 4.5: Controlled Experiments

You now have one trained model and one number. That is not enough to understand anything.

A single accuracy score cannot tell you *why* the model performs the way it does, or which of your
choices mattered. The way to find out is a **controlled experiment**: change exactly one thing,
hold everything else fixed, and measure the difference.

You will run four of them:

| # | What changes | What it reveals |
|---|--------------|-----------------|
| 1 | Learning rate | Why step size has *two* different failure modes |
| 2 | Activation functions removed | Where a deep network's power actually comes from |
| 3 | Training set size and epochs | What overfitting looks like on a real curve |
| 4 | Batch size | The speed / gradient-quality trade-off |

**Important:** every experiment rebuilds the model from scratch with the same seed. If you reused an
already-trained model, you would be measuring the leftover training, not the variable you changed.

These experiments train on a **subset** of the data to keep runtimes short. That is a deliberate
trade-off: smaller samples mean noisier estimates, which is itself worth noticing.

In [ ]:
# Shared experiment helpers. Read these before running the experiments below.

QUICK_N = 6000   # training subset size used by the fast experiments
EVAL_N = 3000    # validation subset size used for quick scoring

quick_train = Subset(train_dataset, range(QUICK_N))
quick_val = Subset(val_dataset, range(min(EVAL_N, len(val_dataset))))
quick_val_loader = DataLoader(quick_val, batch_size=256, shuffle=False)
quick_loader = DataLoader(quick_train, batch_size=BATCH_SIZE, shuffle=True)


def fresh_model(hidden_dim=None, seed=SEED, model_class=None):
    """Build a brand-new, untrained model with reproducible initial weights."""
    torch.manual_seed(seed)
    hidden_dim = HIDDEN_DIM if hidden_dim is None else hidden_dim
    model_class = MLP if model_class is None else model_class
    return model_class(hidden_dim).to(device)


def train_model(model, loader, lr, epochs, val_loader=None, verbose=False):
    """Train `model` and optionally record train/val accuracy after each epoch."""
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr)
    history = {'train': [], 'val': []}

    for ep in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = crit(model(images), labels)
            loss.backward()
            opt.step()

        if val_loader is not None:
            _, tr_acc = evaluate(model, loader, crit)
            _, va_acc = evaluate(model, val_loader, crit)
            history['train'].append(tr_acc)
            history['val'].append(va_acc)
            if verbose:
                print(f'  epoch {ep+1:>3} | train {tr_acc:.4f} | val {va_acc:.4f}')

    return history


def quick_score(model):
    """Validation accuracy on the quick evaluation subset."""
    _, acc = evaluate(model, quick_val_loader, nn.CrossEntropyLoss())
    return acc


print(f'Experiment training subset: {len(quick_train)} images')
print(f'Experiment validation subset: {len(quick_val)} images')


### Experiment 1: Learning Rate

The learning rate controls how large a step gradient descent takes each update. Your notebook used
one value. Here you will try four, spanning five orders of magnitude, and watch two *different*
kinds of failure appear at the two extremes.

In [ ]:
lr_results = {}

print('Learning rate sweep (1 epoch each, identical starting weights)')
for lr in [1.0, 0.1, 0.001, 0.00001]:
    m = fresh_model()
    train_model(m, quick_loader, lr=lr, epochs=1)
    acc = quick_score(m)
    lr_results[lr] = acc
    print(f'  lr={lr:<10} val_acc={acc:.4f}')


**Question 1.** The largest and the smallest learning rates both score badly, but they are *not*
failing for the same reason. Explain what `lr=1.0` does to the weight updates, and what `lr=0.00001`
does. Which of the two would improve if you simply trained for many more epochs, and why?

Answer in the next cell (3-5 sentences).

In [ ]:
exp1_answer = ''  # TODO: Your answer to Question 1


### Experiment 2: Removing the Activation Functions

Your MLP has ReLU activations between its linear layers. This experiment removes them and changes
nothing else: same layer sizes, same parameter count, same learning rate, same data.

If the activations were merely a formality, the two models would score the same.

In [ ]:
class LinearOnly(nn.Module):
    """Identical shape to MLP, but with every activation function removed."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.network(x)


EXP2_EPOCHS = 3

mlp_model = fresh_model()
train_model(mlp_model, quick_loader, lr=LEARNING_RATE, epochs=EXP2_EPOCHS)
mlp_acc = quick_score(mlp_model)

linear_model = fresh_model(model_class=LinearOnly)
train_model(linear_model, quick_loader, lr=LEARNING_RATE, epochs=EXP2_EPOCHS)
linear_acc = quick_score(linear_model)

mlp_params = sum(p.numel() for p in mlp_model.parameters())
linear_params = sum(p.numel() for p in linear_model.parameters())

linear_vs_mlp = {
    'mlp_acc': mlp_acc,
    'linear_acc': linear_acc,
    'mlp_params': mlp_params,
    'linear_params': linear_params,
}

print(f'With ReLU:    val_acc={mlp_acc:.4f}  ({mlp_params:,} parameters)')
print(f'Without ReLU: val_acc={linear_acc:.4f}  ({linear_params:,} parameters)')
print(f'Difference:   {mlp_acc - linear_acc:+.4f}')


**Question 2.** The two models have nearly the same number of parameters, yet one is clearly worse.

Show algebraically why stacking three linear layers with no activation between them is equivalent to a
**single** linear layer. Start from $h_1 = W_1 x + b_1$ and $h_2 = W_2 h_1 + b_2$, substitute, and
identify the single equivalent weight matrix and bias.

Then state in one sentence what this tells you about where a deep network's expressive power comes from.

Answer in the next cell.

In [ ]:
exp2_answer = ''  # TODO: Your answer to Question 2 (include the algebra)


### Experiment 3: Making a Model Overfit

In Part 4 your training and validation curves stayed close together. That is because 48,000 images
is a lot of data for a model this size, so it never needed to memorize.

To *see* overfitting you have to create the conditions for it. Two things cause it here:

1. **A small training set** (500 images) gives the model few enough examples to memorize outright.
2. **Many epochs** give it enough passes to actually do so.

### TODO 4: Set the epoch count

The training set is fixed at 500 images. Your job is to raise `OVERFIT_EPOCHS` until the gap between
the training and validation curves is unmistakable. Start at 30. If the curves have not clearly
separated, increase it and run again.

In [ ]:
# TODO 4: Increase this until the train/validation gap is clearly visible on the plot below.
OVERFIT_EPOCHS = None   # Hint: start at 30, raise it if the curves have not separated

if OVERFIT_EPOCHS is None:
    raise ValueError('Please complete TODO 4 before running this experiment.')

OVERFIT_N = 500
tiny_train = Subset(train_dataset, range(OVERFIT_N))
tiny_loader = DataLoader(tiny_train, batch_size=32, shuffle=True)

overfit_model = fresh_model()
print(f'Training on only {OVERFIT_N} images for {OVERFIT_EPOCHS} epochs...')
overfit_history = train_model(
    overfit_model, tiny_loader, lr=LEARNING_RATE,
    epochs=OVERFIT_EPOCHS, val_loader=quick_val_loader
)

overfit_train_accs = overfit_history['train']
overfit_val_accs = overfit_history['val']
final_gap = overfit_train_accs[-1] - overfit_val_accs[-1]

print(f'Final train acc: {overfit_train_accs[-1]:.4f}')
print(f'Final val acc:   {overfit_val_accs[-1]:.4f}')
print(f'Generalization gap: {final_gap:.4f}')
if final_gap <= 0.05:
    print('Gap is still small. Increase OVERFIT_EPOCHS and run this cell again.')


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(overfit_train_accs) + 1), overfit_train_accs, label='Train Accuracy')
plt.plot(range(1, len(overfit_val_accs) + 1), overfit_val_accs, label='Validation Accuracy')

best_epoch = int(np.argmax(overfit_val_accs)) + 1
plt.axvline(best_epoch, color='red', linestyle='--', alpha=0.7,
            label=f'Best val epoch ({best_epoch})')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title(f'Overfitting on {OVERFIT_N} training images')
plt.legend()
plt.grid(True)
plt.show()

print(f'Validation accuracy peaked at epoch {best_epoch} '
      f'({max(overfit_val_accs):.4f}) and ended at {overfit_val_accs[-1]:.4f}.')


**Question 3.** Look at the plot.

(a) Roughly which epoch do the two curves begin to separate?  
(b) Training accuracy approaches 1.0. Why is that *not* evidence that the model is good?  
(c) You trained on 500 images here and 48,000 images in Part 4, with the same architecture. Explain why
the smaller set produced overfitting that the larger one did not.  
(d) The red line marks the best validation epoch. Describe how you could use that information to decide
when to stop training.

Answer in the next cell.

In [ ]:
exp3_answer = ''  # TODO: Your answer to Question 3, parts (a) through (d)


### Experiment 4: Batch Size

The batch size sets how many examples are averaged together before each weight update.

This connects directly to the probability unit. The gradient computed from a batch of size $n$ is a
**sample mean**: an estimate of the true gradient over the whole dataset, computed from $n$ observations.
Everything you learned about how sample means behave as $n$ grows applies here.

In [ ]:
batch_results = {}

print('Batch size sweep (1 epoch each)')
for bs in [8, 64, 512]:
    loader = DataLoader(quick_train, batch_size=bs, shuffle=True)
    m = fresh_model()

    start = time.time()
    train_model(m, loader, lr=LEARNING_RATE, epochs=1)
    elapsed = time.time() - start

    acc = quick_score(m)
    updates = len(loader)
    batch_results[bs] = {'val_acc': acc, 'seconds': elapsed, 'weight_updates': updates}

    print(f'  batch_size={bs:<5} val_acc={acc:.4f}  time={elapsed:>5.1f}s  updates={updates}')


**Question 4.** Every run above saw all 6,000 images exactly once, but performed a very different
number of weight updates.

(a) Explain the relationship between batch size and the number of weight updates per epoch.  
(b) The gradient from a batch of size $n$ is a sample mean estimate of the true gradient. What happens
to the *variance* of that estimate as $n$ increases?  
(c) Given (a) and (b), explain the trade-off: why might a very large batch train faster per epoch but
still reach worse accuracy in the same number of epochs?

Answer in the next cell.

In [ ]:
exp4_answer = ''  # TODO: Your answer to Question 4, parts (a) through (c)


---
### Optional Challenge: How Much Does the Seed Matter?

**This section is optional and is not required for full credit.** Attempt it if you want to go further.

Every experiment above used a single fixed seed, so every comparison rests on a single run each.
That raises a question worth taking seriously: how much of a difference between two runs is real,
and how much is noise?

Train the same architecture with five different seeds, then report the mean and standard deviation of
validation accuracy.

Then answer this: suppose Model A scores 97.2% and Model B scores 97.4%, each from a single run.
Given the standard deviation you measured, is B actually better than A? What would you need to do to
find out?

In [ ]:
# OPTIONAL. Leave this cell unchanged if you are skipping the challenge.
seed_accs = []

# Uncomment to run:
# for s in [0, 1, 2, 3, 4]:
#     m = fresh_model(seed=s)
#     train_model(m, quick_loader, lr=LEARNING_RATE, epochs=1)
#     seed_accs.append(quick_score(m))
#     print(f'  seed={s} val_acc={seed_accs[-1]:.4f}')
#
# print(f'mean={np.mean(seed_accs):.4f}  std={np.std(seed_accs):.4f}')

seed_answer = ''  # Optional: your answer to the Model A vs Model B question


---
## Part 5: Final Evaluation & Reflections

Evaluate on test data only **once**, after all experiments are finished.

Note that the experiments above deliberately never touched `test_loader`. Every comparison used the
validation set. That is the discipline the split exists to enforce.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'Final Test Accuracy: {test_acc:.4f}')


### Final Reflection

Answer both questions in the cell below.

1. **Validation Set Purpose:** You made many decisions in Part 4.5 by comparing validation scores.
   Explain what would have gone wrong if you had used the test set for those comparisons instead.
   Why would the final number you report no longer mean what it claims to mean?
2. **What You Would Do Next:** Based on your four experiments, name the single change most likely to
   improve your test accuracy, and explain which experimental result led you to that conclusion.

In [ ]:
part5_reflection = '''
1. Validation set purpose:


2. What you would do next:

'''
print('Ensure reflection answers are filled in before submission.')


---
## Export Your Results

Run this cell **last**, after every cell above has been run successfully. It writes `results.json`
next to this notebook.

**Upload both the notebook and `results.json` to Gradescope.**

In [ ]:
# The optional seed challenge may have been skipped; fall back to an empty list.
seed_accs = seed_accs if 'seed_accs' in dir() else []

results = {
    'test_accuracy': float(test_acc),
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'hidden_dim': HIDDEN_DIM,
    },
    'train_accs': [float(a) for a in train_accs],
    'val_accs': [float(a) for a in val_accs],
    'lr_sweep': {str(k): float(v) for k, v in lr_results.items()},
    'linear_vs_mlp': {
        'mlp_acc': float(linear_vs_mlp['mlp_acc']),
        'linear_acc': float(linear_vs_mlp['linear_acc']),
        'mlp_params': int(linear_vs_mlp['mlp_params']),
        'linear_params': int(linear_vs_mlp['linear_params']),
    },
    'overfit_experiment': {
        'epochs': int(OVERFIT_EPOCHS),
        'train_size': int(OVERFIT_N),
        'final_train_acc': float(overfit_train_accs[-1]),
        'final_val_acc': float(overfit_val_accs[-1]),
        'train_curve': [float(a) for a in overfit_train_accs],
        'val_curve': [float(a) for a in overfit_val_accs],
    },
    'batch_sweep': {
        str(k): {
            'val_acc': float(v['val_acc']),
            'seconds': float(v['seconds']),
            'weight_updates': int(v['weight_updates']),
        }
        for k, v in batch_results.items()
    },
    'seed_study': {
        'accs': [float(a) for a in seed_accs],
        'mean': float(np.mean(seed_accs)) if seed_accs else None,
        'std': float(np.std(seed_accs)) if seed_accs else None,
    },
}

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Wrote results.json. Upload this file along with your notebook.')
print(json.dumps(results['hyperparameters'], indent=2))
print(f"test_accuracy: {results['test_accuracy']:.4f}")
